In [1]:
from langchain_openrouter import ChatOpenRouter
import base64
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os

load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

# 2. モデルを初期化
model = ChatOpenRouter(
    model="openai/gpt-5.4-mini",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)


def encode_image(img_path):
    """ローカル画像を Base64 エンコード文字列に変換し、テキスト内に画像データを埋め込みやすくする"""
    with open(img_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")


# 画像パス
img_path = "image_test.jpg"
img_typ = img_path.split(".")[-1]
# 画像の base64 エンコード文字列を取得
base64_image = encode_image(img_path)

response = model.invoke(
    [
        HumanMessage(
            content_blocks=[
                {'type': 'text', 'text': 'この画像には何が写っていますか？'},
                {
                    'type': 'image',
                    'base64': base64_image,
                    'mime_type': 'image/' + img_typ,
                }
            ]
            # content=[
            #     {'type': 'text', 'text': 'この画像には何が写っていますか？'},
            #     {
            #         'type': 'image_url',
            #         "image_url": base64_image,
            #     }
            # ]
        )
    ]
)
print(response.content)

木の上にいる、黒と白の毛がふわふわした動物が写っています。  
見た目からすると、**ジャイアントパンダ**が木に登っているように見えます。


# 2、content_blocksの使用

例1：入力のフォーマット化

In [2]:
from langchain.messages import HumanMessage
import os
import base64
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

# 2. モデルを初期化
model = ChatOpenRouter(
    model="openai/gpt-5.4-mini",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)


def encode_image(img_path):
    """ローカル画像を Base64 エンコード文字列に変換し、テキスト内に画像データを埋め込みやすくする"""
    with open(img_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")


# 画像パス
img_path = "image_test.jpg"
img_typ = img_path.split(".")[-1]
# 画像の base64 エンコード文字列を取得
base64_image = encode_image(img_path)

response = model.invoke(
    [
        HumanMessage(
            content_blocks=[
                {'type': 'text', 'text': 'この画像には何が写っていますか？'},
                {
                    'type': 'image',
                    'base64': base64_image,
                    'mime_type': 'image/' + img_typ,
                }
            ]
            # content=[
            #     {'type': 'text', 'text': 'この画像には何が写っていますか？'},
            #     {
            #         'type': 'image_url',
            #         "image_url": base64_image,
            #     }
            # ]
        )
    ]
)
print(response.content)

木の枝にしがみついている、ふわふわした黒白の動物が写っています。  
見た目からすると、**パンダのように見える動物**ですが、**お尻側が写っていて顔は見えていません**。  
周囲には緑の葉が茂っています。


## 比較として

In [3]:
import base64
from langchain.messages import HumanMessage

from dotenv import load_dotenv

load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

# 2. モデルを初期化
model = ChatOpenRouter(
    model="anthropic/claude-haiku-4.5",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)


def encode_image(img_path):
    """ローカル画像を Base64 エンコードの Data URI 文字列に変換し、テキスト内に画像データを埋め込みやすくする"""
    with open(img_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")


# 画像パス
img_path = "image_test.jpg"

# 画像の base64 エンコード文字列を取得
base64_image = encode_image(img_path)
img_typ = img_path.split(".")[-1]
response = model.invoke(
    [
        # 従来の書き方：content を使用。読み込み失敗
        # HumanMessage(
        #     content=[
        #         {'type': 'text', 'text': 'この画像には何が写っていますか？'},
        #         {
        #             'type': 'image_url',
        #             "image_url": base64_image,
        #         }
        #     ]
        # )

        # 推奨される統一的な書き方
        HumanMessage(
            content_blocks=[
                {'type': 'text', 'text': 'この画像には何が写っていますか？'},
                {
                    'type': 'image',
                    'base64': base64_image,
                    'mime_type': 'image/'+img_typ,
                }
            ]
        )
    ]
)
print(response.content)

この画像には**シフアカ（またはコイフアカ）**という**キツネザル**が写っています。

マダガスカル原産のこの霊長類の特徴は：

- **白と黒の毛色**：白いボディに黒い顔と耳
- **ふさふさした尾**：バランスを取るために使われます
- **樹上生活**：写真のように木の枝にしがみついて生活しています
- **独特の姿勢**：腕を広げた特徴的な動きで知られています

シフアカは夜行性で、マダガスカルの熱帯雨林に生息しており、果実や葉を食べます。群れで生活し、独特の鳴き声で知られています。


例2：出力のフォーマット化

In [4]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print

load_dotenv()

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    extra_body={"thinking": {"type": "enabled"}},
)

response = model.invoke("こんにちは、一言で答えてください")
print('=' * 20, '-> response <-', '=' * 20)
print(response)
print('=' * 20, '-> response.content <-', '=' * 20)
print(response.content)
print('=' * 20, '-> response.content_blocks <-', '=' * 20)
print(response.content_blocks)

==================== -> response <- ====================

AIMessage(
    content='こんにちは',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'We need answer in Japanese. "こんにちは、一言で答えてください" means "Hello, please 
answer in one word." Need respond with one word. Perhaps "はい" or "こんにちは"? Need one word. Maybe "どうぞ" no. 
Let\'s comply: "こんにちは" is greeting. But "一言で答えてください" asks answer in one word. Could answer "はい" 
but weird. Maybe "こんにちは" is one word (aisatsu). I\'ll respond with "こんにちは" only.'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 128,
            'prompt_tokens': 94,
            'total_tokens': 222,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 122,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 94
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402',
        'id': '5e6558cb-5a3c-4829-90c2-0cbe846105de',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019fbb80-a9a0-7d13-b1b9-9ed39db11006-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 94,
        'output_tokens': 128,
        'total_tokens': 222,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 122}
    }
)

==================== -> response.content <- ====================

こんにちは

==================== -> response.content_blocks <- ====================

[
    {
        'type': 'reasoning',
        'reasoning': 'We need answer in Japanese. "こんにちは、一言で答えてください" means "Hello, please answer in
one word." Need respond with one word. Perhaps "はい" or "こんにちは"? Need one word. Maybe "どうぞ" no. Let\'s 
comply: "こんにちは" is greeting. But "一言で答えてください" asks answer in one word. Could answer "はい" but 
weird. Maybe "こんにちは" is one word (aisatsu). I\'ll respond with "こんにちは" only.'
    },
    {'type': 'text', 'text': 'こんにちは'}
]